# mospi-microdata — Test Notebook

Tests the full Python API: catalog search, metadata from microdata.gov.in, data querying, guardrails.

In [ ]:
from mospi_microdata import Catalog, NetworkError, DataNotFoundError, InvalidTableError, QueryError

## 1. Catalog Discovery
Search the official NADA catalog at microdata.gov.in

In [ ]:
catalog = Catalog(data_dir="./data")
results = catalog.search(keyword="industries")
print(f"Found {len(results)} results\n")
for r in results[:5]:
    print(f"  {r['idno']}: {r['title']}")

## 2. Load ASI 2023-24

In [ ]:
asi = catalog.load("ASI", "2023-24")
print("Loaded ASI 2023-24")

## 3. Study Metadata
All metadata comes from microdata.gov.in — nothing hardcoded

In [ ]:
meta = asi.metadata()
print(f"Title: {meta['title']}")
print(f"Keywords: {meta['keywords']}")
print(f"Producers: {meta['producers']}")
print(f"Total variables: {meta['varcount']}")
print(f"\nAbstract (first 300 chars):\n{meta['abstract'][:300]}...")

## 4. Data Files

In [ ]:
import pandas as pd

files = asi.files()
pd.DataFrame(files)

## 5. Variables in a File

In [ ]:
vars_e = asi.variables("block_e")
pd.DataFrame(vars_e)

In [ ]:
vars_a = asi.variables("block_a")
pd.DataFrame(vars_a)

## 6. Variable Detail
Includes value labels, summary stats, and ranges — all from NADA

In [ ]:
# Continuous variable
detail = asi.variable("block_e", "EI6")
print(f"Name: {detail['name']}")
print(f"Label: {detail['label']}")
print(f"Type: {detail['type']}")
print(f"Range: {detail.get('range')}")
print(f"Summary stats: {detail.get('summary_stats')}")

In [ ]:
# Categorical variable with value labels
detail_a9 = asi.variable("block_a", "a9")
print(f"Name: {detail_a9['name']}")
print(f"Label: {detail_a9['label']}")
print(f"Value labels: {detail_a9.get('value_labels')}")
print(f"Frequencies: {detail_a9.get('frequencies')}")

## 7. Join Map

In [ ]:
jm = asi.join_map()
for table, key in jm.items():
    print(f"  {table} → {key}")

## 8. Sample Data

In [ ]:
asi.sample("block_a", n=5)

In [ ]:
asi.sample("block_j", n=5)

## 9. Load File

In [ ]:
# Load specific columns
df = asi.load_file("block_e", columns=["AE01", "EI1", "EI6"])
print(f"Shape: {df.shape}")
df.head()

## 10. SQL Queries
### Weighted total employment

In [ ]:
asi.query("""
    SELECT ROUND(SUM(e.EI6 * a.mult)) as total_workers
    FROM block_e e JOIN block_a a ON e."AE01" = a.a1
    WHERE e.EI1 = 10 AND a.mult > 0
""")

### Employment by NIC 2-digit industry

In [ ]:
df_nic = asi.query("""
    SELECT LEFT(a.a5, 2) as nic2, ROUND(SUM(e.EI6 * a.mult)) as workers
    FROM block_e e JOIN block_a a ON e."AE01" = a.a1
    WHERE e.EI1 = 10 AND a.mult > 0
    GROUP BY nic2 ORDER BY workers DESC
""")
df_nic.head(10)

### Total output by sector (Rural/Urban)

In [ ]:
asi.query("""
    SELECT
        a.a9 as sector,
        COUNT(DISTINCT a.a1) as factories,
        ROUND(SUM(a.costop * a.mult)) as weighted_cost_of_production
    FROM block_a a
    WHERE a.mult > 0
    GROUP BY a.a9
""")

### Average wages per worker by industry

In [ ]:
asi.query("""
    SELECT
        LEFT(a.a5, 2) as nic2,
        ROUND(SUM(e.EI8 * a.mult) / NULLIF(SUM(e.EI6 * a.mult), 0)) as avg_wage_per_worker
    FROM block_e e
    JOIN block_a a ON e."AE01" = a.a1
    WHERE e.EI1 = 10 AND a.mult > 0 AND e.EI6 > 0
    GROUP BY nic2
    ORDER BY avg_wage_per_worker DESC
""")

## 11. Guardrails
### SQL guardrails — blocks destructive queries

In [ ]:
try:
    asi.query("DROP TABLE block_a")
except QueryError as e:
    print(f"Blocked: {e}")

try:
    asi.query("DELETE FROM block_a WHERE 1=1")
except QueryError as e:
    print(f"Blocked: {e}")

### Table name validation — blocks access to non-survey tables

In [ ]:
try:
    asi.sample("not_a_real_table", 5)
except InvalidTableError as e:
    print(f"Blocked: {e}")

try:
    asi.load_file("information_schema.tables")
except InvalidTableError as e:
    print(f"Blocked: {e}")

### Row cap — limit prevents memory blowup

In [ ]:
df_limited = asi.query("SELECT * FROM block_a", limit=10)
print(f"Requested limit=10, got {len(df_limited)} rows (not all 68,641)")
df_limited.head()

### Missing data file — checked before network call

In [ ]:
try:
    catalog.load("ASI", "1999-00")
except DataNotFoundError as e:
    print(f"Caught (no network call made): {e}")